# Estudo de Caso 9.6 — Condensador de Vapor para UTE a Gás: Dimensionamento Térmico e Impacto Ambiental

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 9 — Trocadores de Calor |
| **Nível** | Ambos (Graduação e Pós-Graduação) |
| **Tempo estimado** | 90 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | LMTD com fluido em mudança de fase + ε-NTU (validação) |
| **Bibliografia** | Incropera et al. (2017), Cap. 11; Kakaç, Liu & Majumdar (2012), Cap. 3; Gobbi (s.d.), Cap. 2 |

---

## 📋 Resumo

Dimensionamos o condensador de vapor de uma usina termelétrica (UTE) a gás natural com potência elétrica nominal de 100 MW e eficiência térmica de 40%. O condensador utiliza água captada de um rio como fluido de resfriamento para liquefazer o vapor de exaustão da turbina, fechando o ciclo Rankine. O dimensionamento utiliza o método LMTD para fluido em temperatura constante (condensação), com validação cruzada pelo método ε-NTU. Uma análise de sensibilidade investiga o impacto da temperatura do rio e da incrustação no desempenho térmico, além da avaliação do impacto ambiental do lançamento de água aquecida.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Aplicar** o balanço de energia em ciclo termodinâmico real (ciclo Rankine)
2. **Calcular** a LMTD para trocador com fluido em temperatura constante (condensação)
3. **Dimensionar** a área de troca térmica e a configuração física do condensador
4. **Verificar** a velocidade da água nos tubos e o regime de escoamento
5. **Analisar** a sensibilidade do projeto a variações de temperatura do rio e incrustação
6. **Avaliar** o impacto ambiental do lançamento de efluente térmico em corpos d'água
7. **Comparar** resultados obtidos por métodos distintos (LMTD e ε-NTU)

---
## ⚙️ Seção 0: Configuração Inicial

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL — Importação de bibliotecas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
import time

# Configuração de estilo dos gráficos (padrão do guia)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Ambiente configurado com sucesso')
print(f'📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}')

---
## 🌊 Seção 1: Parâmetros do Condensador

### Contexto Industrial

Uma **usina termelétrica (UTE)** a gás natural opera com um ciclo Rankine modificado. O gás natural é queimado em uma câmara de combustão, gerando gases quentes que aquecem água em uma caldeira, produzindo vapor em alta pressão. Esse vapor expande-se através de uma turbina, gerando eletricidade. Após passar pela turbina, o vapor precisa ser **condensado** (liquefeito) para retornar à caldeira, fechando o ciclo.

O **condensador** é o trocador de calor responsável por essa condensação:
- **Fluido quente:** vapor d'água em baixa pressão (condensando a temperatura constante)
- **Fluido frio:** água captada de um rio (aquecendo sem mudança de fase)

> 💡 **Observação:** A eficiência do condensador impacta diretamente a eficiência global da usina: quanto menor a temperatura de condensação, maior o trabalho útil extraído da turbina.

### Tabela de Parâmetros

| Parâmetro | Símbolo | Valor | Unidade |
|:----------|:-------:|------:|:-------:|
| **Usina Termelétrica** | | | |
| Potência elétrica | $W_{el}$ | 100 | MW |
| Eficiência térmica | $\eta$ | 0,40 | – |
| **Vapor (lado casco)** | | | |
| Temperatura de condensação | $T_{vap}$ | 50 | °C |
| Pressão de condensação | $p_{vap}$ | ≈ 12,3 | kPa |
| Calor latente de vaporização | $h_{fg}$ | 2383 | kJ/kg |
| **Água de resfriamento (lado tubos)** | | | |
| Temperatura de entrada | $T_{agua,e}$ | 22 | °C |
| Temperatura de saída (máx.) | $T_{agua,s}$ | 32 | °C |
| Calor específico da água | $c_{p,agua}$ | 4180 | J/(kg·K) |
| Densidade da água | $\rho_{agua}$ | 998 | kg/m³ |
| **Condensador** | | | |
| Coeficiente global | $U$ | 1800 | W/(m²·K) |
| Diâmetro externo dos tubos | $D_e$ | 20 | mm |
| Diâmetro interno dos tubos | $D_i$ | 18 | mm |
| Comprimento dos tubos | $L$ | 6,0 | m |
| Número de passadas | $n_p$ | 2 | – |
| Material dos tubos | – | Cu-Ni | – |

In [ ]:
# ============================================================
# PARÂMETROS DO CONDENSADOR E DA UTE
# ============================================================

# --- Usina Termelétrica ---
W_el = 100.0          # Potência elétrica [MW]
eta = 0.40            # Eficiência térmica do ciclo [-]

# --- Vapor (lado casco - condensação) ---
T_vap = 50.0          # Temperatura de condensação [°C]
p_vap = 12.3          # Pressão de condensação [kPa]
h_fg = 2383.0         # Calor latente de vaporização [kJ/kg]

# --- Água de resfriamento (lado tubos) ---
T_agua_e = 22.0       # Temperatura de entrada da água [°C]
T_agua_s = 32.0       # Temperatura de saída da água [°C]
cp_agua = 4180.0      # Calor específico da água [J/(kg·K)]
rho_agua = 998.0      # Densidade da água [kg/m³]

# --- Condensador ---
U = 1800.0            # Coeficiente global [W/(m²·K)]
D_ext_tubo = 0.020    # Diâmetro externo do tubo [m]
D_int_tubo = 0.018    # Diâmetro interno do tubo [m]
L_tubo = 6.0          # Comprimento do tubo [m]
n_passes = 2          # Número de passadas [-]

print('📏 Parâmetros do Condensador:')
print(f'  • UTE: W_el={W_el} MW, η={eta*100:.0f}%')
print(f'  • Vapor: T_vap={T_vap}°C, h_fg={h_fg} kJ/kg')
print(f'  • Água: T_e={T_agua_e}°C, T_s={T_agua_s}°C, ΔT={T_agua_s-T_agua_e}°C')
print(f'  • Condensador: U={U} W/(m²·K)')
print(f'  • Tubos: Ø{D_ext_tubo*1000:.0f}×{D_int_tubo*1000:.0f} mm, L={L_tubo} m, Cu-Ni')

---
## 🔬 Seção 2: Verificação — Balanço Energético da UTE

Antes do dimensionamento, verificamos o balanço energético da usina.

### Passo 1: Calor Fornecido pela Caldeira

Pela definição de eficiência térmica:

$$\eta = \frac{W_{el}}{Q_{caldeira}} \quad \Rightarrow \quad Q_{caldeira} = \frac{W_{el}}{\eta}$$

### Passo 2: Calor Rejeitado no Condensador

Pelo balanço de energia global da UTE:

$$Q_{cond} = Q_{caldeira} - W_{el}$$

### Critério de Verificação

$$\frac{Q_{cond}}{Q_{caldeira}} = 1 - \eta$$

Para $\eta = 40\%$, esperamos que 60% do calor seja rejeitado.

> 📖 **Referência:** Incropera et al. (2017), Cap. 11, Seção 11.2.

In [ ]:
# ============================================================
# BALANÇO ENERGÉTICO DA UTE
# ============================================================

t_inicio = time.time()

# Cálculos
Q_caldeira = W_el / eta           # MW
Q_cond = Q_caldeira - W_el        # MW
Q_cond_W = Q_cond * 1e6           # W

# Vazões
m_vap = Q_cond_W / (h_fg * 1e3)                       # kg/s
m_agua = Q_cond_W / (cp_agua * (T_agua_s - T_agua_e)) # kg/s
V_agua = m_agua / rho_agua                            # m³/s

t_cpu_balanco = time.time() - t_inicio

print('=' * 60)
print('  BALANÇO ENERGÉTICO DA UTE')
print('=' * 60)
print(f'\n📊 Potências:')
print(f'  • W_el = {W_el:.0f} MW (eletricidade)')
print(f'  • Q_caldeira = {Q_caldeira:.0f} MW (calor fornecido)')
print(f'  • Q_cond = {Q_cond:.0f} MW (calor rejeitado)')
print(f'\n📊 Vazões:')
print(f'  • Vapor: ṁ_vap = {m_vap:.1f} kg/s')
print(f'  • Água: ṁ_agua = {m_agua:.0f} kg/s')
print(f'  • Água: V̇_agua = {V_agua:.2f} m³/s ({V_agua*3600:.0f} m³/h)')
print(f'\n✅ Verificação: Q_cond/Q_caldeira = {Q_cond/Q_caldeira*100:.0f}% (= 1-η = {(1-eta)*100:.0f}%)')
print(f'⏱️ Tempo de CPU: {t_cpu_balanco:.6f} s')

> 💡 **Observação:**
> - A UTE rejeita 150 MW no condensador — o equivalente a 60% da energia do combustível
> - Isso ilustra a limitação da Segunda Lei da Termodinâmica
> - A vazão de água de 3,60 m³/s equivale a 3.600 litros por segundo

---
## 📐 Seção 3: Formulação Matemática

### 3.1 LMTD para Fluido em Temperatura Constante

Como o vapor condensa a **temperatura constante** ($T_{vap} = 50°C$), o perfil de temperatura é característico de um fluido em mudança de fase. Nesse caso, a LMTD é a mesma tanto para configurações em paralelo quanto em contracorrente.

$$\Delta T_{lm} = \frac{\Delta T_1 - \Delta T_2}{\ln(\Delta T_1 / \Delta T_2)}$$

onde:
- $\Delta T_1 = T_{vap} - T_{agua,e}$ [°C]
- $\Delta T_2 = T_{vap} - T_{agua,s}$ [°C]

### 3.2 Área de Troca Térmica

$$A = \frac{Q_{cond}}{U \cdot \Delta T_{lm}}$$

### 3.3 Método ε-NTU (Validação)

Para condensação, $C_c \to \infty$, portanto $C_r = 0$:

$$\varepsilon = 1 - \exp(-NTU), \quad NTU = \frac{U \cdot A}{C_{min}}, \quad C_{min} = \dot{m}_{agua} \cdot c_{p,agua}$$

> 📖 **Referência:** Incropera et al. (2017), Cap. 11, Seções 11.3–11.6.

### 3.4 Coeficiente Global Elevado

O valor $U = 1800$ W/(m²·K) é elevado porque:
- Condensação de vapor: $h \approx 5000$–$15000$ W/(m²·K)
- Água nos tubos: $h \approx 3000$–$6000$ W/(m²·K)
- Resistência dominante: incrustação + parede metálica

---
## ⚡ Seção 4: Critério de Projeto

> 💡 **Observação:** Neste problema de dimensionamento em regime permanente, não se aplica a condição CFL. Os critérios de adequação são:

1. **Velocidade da água nos tubos:** $1{,}5 \leq v \leq 2{,}5$ m/s para evitar incrustação (velocidade mínima) e erosão (velocidade máxima).

2. **Regime de escoamento:** $Re > 4000$ (turbulento) para garantir boa transferência de calor.

3. **Elevação de temperatura da água:** $\Delta T \leq 10°C$ conforme limites ambientais (Resolução CONAMA 357/2005).

4. **Validade da LMTD unificada:** Para fluido em mudança de fase, a LMTD é idêntica para qualquer configuração (paralelo, contracorrente, cruzado).

In [ ]:
# ============================================================
# CRITÉRIO DE PROJETO — Verificação de velocidade e regime
# ============================================================

# Área por tubo
A_tubo = np.pi * D_ext_tubo * L_tubo   # m²

# Número de tubos (estimativa inicial)
N_tubos_est = 9815

# Área de escoamento nos tubos (considerando passadas)
tubos_por_passe = N_tubos_est / n_passes
A_fluxo = tubos_por_passe * (np.pi * D_int_tubo**2 / 4)

# Velocidade da água
v_agua = V_agua / A_fluxo

# Número de Reynolds
nu_agua = 1.0e-6   # m²/s (viscosidade cinemática a ~25°C)
Re = v_agua * D_int_tubo / nu_agua

print('⚡ Critério de Projeto:')
print(f'  • Velocidade da água: v = {v_agua:.2f} m/s')
print(f'  • Faixa recomendada: 1,5–2,5 m/s')
if 1.5 <= v_agua <= 2.5:
    print(f'  ✅ Velocidade dentro da faixa')
elif v_agua < 1.5:
    print(f'  ⚠️ Velocidade ABAIXO — risco de incrustação')
else:
    print(f'  ⚠️ Velocidade ACIMA — risco de erosão')
print(f'  • Número de Reynolds: Re = {Re:.0f}')
if Re > 4000:
    print(f'  ✅ Regime turbulento')
else:
    print(f'  ⚠️ Regime não turbulento')
print(f'  • Elevação de temperatura: ΔT = {T_agua_s - T_agua_e:.0f}°C (limite: ≤10°C)')

---
## 🌪️ Seção 5: Condição de Entrada (Forçamento Térmico)

Neste problema de regime permanente, o forçamento é a **carga térmica do condensador** imposta pelo ciclo termodinâmico:

$$Q_{cond} = 150 \text{ MW (constante)}$$

e a **temperatura de condensação** fixada pela pressão de operação:

$$T_{vap} = 50°C \quad (p \approx 12{,}3 \text{ kPa})$$

> 💡 **Observação:** A temperatura de condensação é fixada pela pressão de exaustão da turbina. Reduzir $T_{vap}$ aumenta o trabalho da turbina, mas exige mais área de troca ou água mais fria.

---
## 🧮 Seção 6: Implementação Numérica

In [ ]:
# ============================================================
# DIMENSIONAMENTO DO CONDENSADOR — Método LMTD
# ============================================================

t_inicio = time.time()

# LMTD
dT1 = T_vap - T_agua_e
dT2 = T_vap - T_agua_s
LMTD = (dT1 - dT2) / np.log(dT1 / dT2)

# Média aritmética (para comparação)
LMTD_arit = (dT1 + dT2) / 2
erro_arit = abs(LMTD_arit - LMTD) / LMTD * 100

# Área de troca
A_req = Q_cond_W / (U * LMTD)   # m²
A_comercial = np.ceil(A_req / 100) * 100

# Número de tubos
N_tubos = A_comercial / A_tubo

t_cpu_lmtd = time.time() - t_inicio

print('=' * 60)
print('  MÉTODO LMTD — DIMENSIONAMENTO')
print('=' * 60)
print(f'\n📊 Diferenças de temperatura:')
print(f'  • ΔT₁ = T_vap - T_agua,e = {dT1:.0f}°C')
print(f'  • ΔT₂ = T_vap - T_agua,s = {dT2:.0f}°C')
print(f'  • Razão ΔT₁/ΔT₂ = {dT1/dT2:.2f}')
print(f'\n📊 LMTD:')
print(f'  • LMTD = {LMTD:.1f}°C')
print(f'  • Média aritmética = {LMTD_arit:.1f}°C (erro: {erro_arit:.1f}%)')
print(f'\n📊 Área de troca:')
print(f'  • A_requerida = {A_req:.0f} m²')
print(f'  • A_comercial = {A_comercial:.0f} m²')
print(f'  • Nº tubos = {N_tubos:.0f}')
print(f'\n⏱️ Tempo de CPU: {t_cpu_lmtd:.6f} s')

In [ ]:
# ============================================================
# VALIDAÇÃO — Método ε-NTU
# ============================================================

t_inicio = time.time()

# Capacidades térmicas
C_agua = m_agua * cp_agua     # W/K (lado tubos)
# Para condensação: C_vapor → ∞, então C_min = C_agua
C_min = C_agua
C_r = 0.0

# NTU
NTU = U * A_comercial / C_min

# Efetividade para C_r = 0
epsilon = 1 - np.exp(-NTU)

# Calor máximo possível
Q_max = C_min * (T_vap - T_agua_e)

# Calor real
Q_real_NTU = epsilon * Q_max

# Temperatura de saída da água
T_agua_s_NTU = T_agua_e + Q_real_NTU / C_agua

t_cpu_ntu = time.time() - t_inicio

print('=' * 60)
print('  MÉTODO ε-NTU — VALIDAÇÃO')
print('=' * 60)
print(f'\n📊 Capacidades térmicas:')
print(f'  • C_agua = {C_agua/1e6:.2f} MW/K')
print(f'  • C_vapor → ∞ (condensação)')
print(f'  • C_r = {C_r:.2f}')
print(f'\n📊 NTU e Efetividade:')
print(f'  • NTU = U·A/C_min = {NTU:.3f}')
print(f'  • ε = 1-exp(-NTU) = {epsilon:.4f} ({epsilon*100:.1f}%)')
print(f'\n📊 Desempenho:')
print(f'  • Q_max = {Q_max/1e6:.1f} MW')
print(f'  • Q_real = {Q_real_NTU/1e6:.1f} MW')
print(f'  • T_agua_s = {T_agua_s_NTU:.1f}°C')
print(f'\n✅ Verificação: Q_real/Q_cond = {Q_real_NTU/Q_cond_W*100:.1f}%')
print(f'⏱️ Tempo de CPU: {t_cpu_ntu:.6f} s')

> 💡 **Observação:**
> - Os métodos LMTD e ε-NTU fornecem resultados consistentes, validando o dimensionamento
> - Para condensação ($C_r = 0$), a efetividade independe da configuração do trocador
> - O erro da média aritmética (~1,6%) é pequeno neste caso, mas a LMTD é sempre mais rigorosa

---
## 📊 Seção 7: Visualização dos Resultados

In [ ]:
# ============================================================
# GRÁFICOS — Balanço energético e perfil de temperatura
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- (a) Balanço energético da UTE ---
ax = axes[0]
labels = ['Caldeira\n(fornecido)', 'Eletricidade\n(útil)', 'Condensador\n(rejeitado)']
values = [Q_caldeira, W_el, Q_cond]
cores = ['#e74c3c', '#2ecc71', '#3498db']
ax.bar(labels, values, color=cores, edgecolor='black', linewidth=0.8)
for i, v in enumerate(values):
    ax.text(i, v + 3, f'{v:.0f} MW', ha='center', fontweight='bold')
ax.set_ylabel('Potência [MW]')
ax.set_title('(a) Balanço Energético da UTE', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# --- (b) Perfil de temperatura no condensador ---
ax = axes[1]
x = np.linspace(0, 1, 100)
T_vap_prof = np.full_like(x, T_vap)
T_agua_prof = T_agua_e + (T_agua_s - T_agua_e) * x
ax.plot(x, T_vap_prof, 'r-', linewidth=3, label='Vapor (condensando)')
ax.plot(x, T_agua_prof, 'b-', linewidth=3, label='Água (aquecendo)')
ax.fill_between(x, T_agua_prof, T_vap_prof, alpha=0.2, color='orange',
                label=f'LMTD = {LMTD:.1f}°C')
ax.set_xlabel('Posição no trocador')
ax.set_ylabel('Temperatura [°C]')
ax.set_title('(b) Perfil de Temperatura', fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

# --- (c) Comparação de áreas ---
ax = axes[2]
metodos = ['LMTD\n(Requerida)', 'Comercial\n(Arredondada)']
A_vals = [A_req, A_comercial]
cores_bar = ['#2196F3', '#4CAF50']
bars = ax.bar(metodos, A_vals, color=cores_bar, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, A_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:.0f} m²', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Área de troca [m²]')
ax.set_title('(c) Área de Troca Térmica', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - O vapor condensa a temperatura constante (50°C), enquanto a água aquece linearmente de 22°C a 32°C
> - A LMTD de 22,6°C é a força motriz média para a transferência de calor
> - A área de ~3700 m² equivale a aproximadamente metade de um campo de futebol

---
## 📈 Seção 8: Análise Quantitativa

In [ ]:
# ============================================================
# TABELA COMPARATIVA DOS MÉTODOS
# ============================================================

dados_tabela = {
    'Grandeza': [
        'Calor da caldeira',
        'Potência elétrica',
        'Calor rejeitado (Q_cond)',
        'Vazão de vapor',
        'Vazão de água (mássica)',
        'Vazão de água (volumétrica)',
        'ΔT₁ (T_vap - T_agua,e)',
        'ΔT₂ (T_vap - T_agua,s)',
        'LMTD',
        'Coeficiente global U',
        'Área requerida',
        'Área comercial',
        'Número de tubos',
        'NTU (ε-NTU)',
        'Efetividade ε',
        'Velocidade da água'
    ],
    'Valor': [
        f'{Q_caldeira:.0f}',
        f'{W_el:.0f}',
        f'{Q_cond:.0f}',
        f'{m_vap:.1f}',
        f'{m_agua:.0f}',
        f'{V_agua:.2f}',
        f'{dT1:.0f}',
        f'{dT2:.0f}',
        f'{LMTD:.1f}',
        f'{U:.0f}',
        f'{A_req:.0f}',
        f'{A_comercial:.0f}',
        f'{N_tubos:.0f}',
        f'{NTU:.3f}',
        f'{epsilon:.4f}',
        f'{v_agua:.2f}'
    ],
    'Unidade': [
        'MW', 'MW', 'MW', 'kg/s', 'kg/s', 'm³/s',
        '°C', '°C', '°C', 'W/(m²·K)', 'm²', 'm²',
        '–', '–', '–', 'm/s'
    ]
}

df_resumo = pd.DataFrame(dados_tabela)
display(Markdown(df_resumo.to_markdown(index=False)))

---
## 🔍 Seção 9: Análise de Sensibilidade

Investigamos o impacto de dois fatores críticos no desempenho do condensador:
1. **Temperatura da água do rio** (variação sazonal)
2. **Incrustação nos tubos** (degradação ao longo do tempo)

In [ ]:
# ============================================================
# SENSIBILIDADE 1: Temperatura da água do rio
# ============================================================

# Varia T_agua_e mantendo ΔT = 10°C
T_agua_range = np.linspace(18, 32, 15)
A_range = []
LMTD_range = []

for T_e in T_agua_range:
    T_s = T_e + 10.0
    dT1_test = T_vap - T_e
    dT2_test = T_vap - T_s
    if dT2_test <= 0:
        A_range.append(np.nan)
        LMTD_range.append(np.nan)
        continue
    lmtd_test = (dT1_test - dT2_test) / np.log(dT1_test / dT2_test)
    A_test = Q_cond_W / (U * lmtd_test)
    A_range.append(A_test)
    LMTD_range.append(lmtd_test)

A_range = np.array(A_range)
LMTD_range = np.array(LMTD_range)

# Gráfico
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(T_agua_range, LMTD_range, 'o-', color='darkorange', linewidth=2, markersize=7)
ax1.axhline(LMTD, color='gray', linestyle='--', alpha=0.6,
            label=f'Caso base: {LMTD:.1f}°C')
ax1.set_xlabel('Temperatura de entrada da água [°C]')
ax1.set_ylabel('LMTD [°C]')
ax1.set_title('LMTD × Temperatura da água do rio', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.plot(T_agua_range, A_range, 's-', color='steelblue', linewidth=2, markersize=7)
ax2.axhline(A_req, color='gray', linestyle='--', alpha=0.6,
            label=f'Caso base: {A_req:.0f} m²')
ax2.set_xlabel('Temperatura de entrada da água [°C]')
ax2.set_ylabel('Área necessária [m²]')
ax2.set_title('Área × Temperatura da água do rio', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n📊 Sensibilidade à temperatura do rio:')
print(f'  • T_agua,e = 18°C (inverno): A ≈ {A_range[0]:.0f} m²')
print(f'  • T_agua,e = 22°C (normal):  A ≈ {A_range[4]:.0f} m²')
print(f'  • T_agua,e = 28°C (verão):   A ≈ {A_range[10]:.0f} m²')
print(f'  • T_agua,e = 32°C (extremo): A ≈ {A_range[14]:.0f} m²')
aumento = (A_range[10] / A_req - 1) * 100
print(f'\n  ⚠️ Aumento de área no verão (28°C): {aumento:.0f}%')

In [ ]:
# ============================================================
# SENSIBILIDADE 2: Incrustação após 1 ano
# ============================================================

Rf = 0.0002   # Fator de incrustação [m²·K/W]

# Novo coeficiente global
U_proj = 1.0 / (1.0/U + Rf)

# Nova carga térmica máxima (mantendo área e LMTD)
Q_cond_novo = U_proj * A_comercial * LMTD   # W

print('=' * 60)
print('  SENSIBILIDADE — INCRUSTAÇÃO')
print('=' * 60)
print(f'\n📊 Coeficiente global:')
print(f'  • U limpo = {U:.0f} W/(m²·K)')
print(f'  • U incrustado = {U_proj:.0f} W/(m²·K)')
print(f'  • Redução: {(1 - U_proj/U)*100:.1f}%')
print(f'\n📊 Desempenho com incrustação:')
print(f'  • Q_max limpo = {Q_cond_W/1e6:.0f} MW')
print(f'  • Q_max incrustado = {Q_cond_novo/1e6:.1f} MW')
if Q_cond_novo/1e6 < Q_cond:
    deficit = Q_cond - Q_cond_novo/1e6
    print(f'  ⚠️ DÉFICIT de {deficit:.1f} MW → usina NÃO mantém 100 MW')
    print(f'  • Solução: limpeza periódica dos tubos')
else:
    print(f'  ✅ Usina ainda mantém 100 MW')

> 💡 **Observação:**
> - Em período de estiagem (T_agua,e ≈ 28°C), a área necessária cresce ~57%
> - A incrustação (R_f = 0,0002 m²·K/W) reduz U em ~26%, podendo comprometer a potência nominal
> - Soluções: limpeza periódica dos tubos, superdimensionamento inicial, ou torres de resfriamento

---
## 🌍 Impacto Ambiental

O lançamento de água aquecida no rio exige avaliação ambiental conforme a Resolução CONAMA 357/2005.

In [ ]:
# ============================================================
# IMPACTO AMBIENTAL — Comparação com vazão de rios
# ============================================================

rios = {
    'Rio Macaé (RJ)': 6.0,
    'Rio Tietê (SP)': 100.0,
    'Rio Paraíba do Sul': 500.0,
    'Rio São Francisco': 700.0,
    'Rio Amazonas': 160000.0
}

print('=' * 60)
print('  IMPACTO AMBIENTAL')
print('=' * 60)
print(f'\n📊 Vazão de captação: {V_agua:.2f} m³/s')
print(f'📊 Elevação de temperatura: {T_agua_s - T_agua_e:.0f}°C')
print(f'\n📊 Comparação com vazões de rios:')
print(f'{"Rio":<30} {"Vazão (m³/s)":>12} {"Fração captada":>15}')
print('-' * 60)
for nome, vazao in rios.items():
    fracao = V_agua / vazao * 100
    print(f'{nome:<30} {vazao:>12.1f} {fracao:>14.2f}%')

print(f'\n📋 Legislação (CONAMA 357/2005):')
print(f'  • Elevação máxima: 3°C acima da temperatura natural')
print(f'  • Zona de mistura: autorizada mediante EIA/RIMA')
print(f'\n💡 Alternativas de mitigação:')
print(f'  1. Torres de resfriamento')
print(f'  2. Lagos de resfriamento')
print(f'  3. Difusores de lançamento')
print(f'  4. Reuso da água aquecida')

---
## 💡 Seção 10: Discussão

### Interpretação Física dos Resultados

1. **Rejeição térmica dominante:** A UTE rejeita 60% da energia do combustível no condensador, ilustrando a limitação da Segunda Lei da Termodinâmica. Para cada 250 MW de calor gerado, apenas 100 MW se convertem em eletricidade.

2. **Coeficiente global elevado:** O valor $U = 1800$ W/(m²·K) é elevado porque a condensação de vapor possui coeficientes de película de 5000 a 15000 W/(m²·K). A resistência dominante é a incrustação e a parede metálica.

3. **Sensibilidade à temperatura do rio:** O condensador é altamente sensível às condições ambientais. Em período de estiagem, a redução da LMTD exige área significativamente maior, podendo comprometer a operação nominal.

4. **Vazão de água expressiva:** Os 3,60 m³/s de água de resfriamento representam uma captação significativa, exigindo estudos de impacto ambiental e licenciamento.

### Limitações do Modelo

- **Regime permanente:** O modelo não considera transientes (partida, variação de carga, mudanças sazonais bruscas).
- **U constante:** O coeficiente global é assumido constante. Na realidade, varia com a velocidade da água, grau de incrustação e condições de condensação.
- **LMTD unificada:** Assume que a condensação é completa e isotérmica. Na prática, pode haver sub-resfriamento do condensado.
- **Geometria simplificada:** O modelo não considera detalhes do arranjo dos tubos, passes e placas tubulares.

### Aplicações Práticas

- **Dimensionamento de condensadores industriais:** O método LMTD é diretamente aplicável ao projeto de condensadores para usinas termelétricas e processos industriais.
- **Estudos de impacto ambiental:** A análise de vazão e elevação de temperatura é essencial para o licenciamento ambiental de usinas.

### Conexão com a Teoria

> 📖 **Referência:** Os métodos LMTD e ε-NTU são apresentados em Incropera et al. (2017), Cap. 11, Seções 11.3–11.6. A análise de impacto ambiental de efluentes térmicos segue Gobbi (s.d.), Cap. 2, sobre balanços de energia em corpos d'água.

---
## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 9](../notebooks/09_Trocadores_Calor.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 9.6

Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*
</div>